## Example 1: Kafka producer setup and message publishing

In [ ]:
from kafka import KafkaProducer, __version__
from kafka.errors import KafkaTimeoutError
from kafka.structs import TopicPartition, OffsetAndMetadata
import json
import socket

# Example 1: Kafka producer setup and message publishing
print("Example 1: Kafka producer setup and message publishing")

print(f"Kafka-python version: {__version__}")

_probe = socket.socket()
_probe.bind(("127.0.0.1", 0))
broker_port = _probe.getsockname()[1]
_probe.close()
producer = KafkaProducer(
    bootstrap_servers=[f'127.0.0.1:{broker_port}'],  # no broker listens here
    value_serializer=lambda v: json.dumps(v).encode('utf-8'),
    key_serializer=lambda k: k.encode('utf-8') if k else None,
    retries=3,
    acks='all',
    compression_type='gzip',
    max_block_ms=1,  # Minimal timeout: fail fast instead of waiting for broker metadata
    api_version=(2, 8, 0)  # Explicit protocol version: no bootstrap round-trip at construction
)
print(f"Producer created: {type(producer).__name__}")
assert producer.config['acks'] == -1 and producer.config['retries'] == 3, producer.config
assert producer.config['compression_type'] == 'gzip', producer.config['compression_type']

# With no broker, send() cannot fetch the topic's metadata and gives up after max_block_ms
try:
    producer.send("user-events", value={"user_id": 123, "action": "login"}, key="user-123")
except KafkaTimeoutError as e:
    print(f"send() with no broker timed out as expected -> {type(e).__name__}")
else:
    raise AssertionError("send() succeeded with no broker listening")

# Demonstrate message structure
topic = "user-events"
sample_messages = [
    {"user_id": 123, "action": "login", "timestamp": "2024-01-15T10:30:00Z"},
    {"user_id": 456, "action": "purchase", "amount": 99.99, "product": "laptop"}
]

print(f"\nMessage examples for topic '{topic}':")
for i, msg in enumerate(sample_messages, 1):
    print(f"  Message {i}: {msg}")

# Show producer configuration options (without accessing actual config)
config_options = {
    "bootstrap_servers": "localhost:9092",
    "acks": "all",
    "retries": 3,
    "compression_type": "gzip",
    "batch_size": 16384,
    "linger_ms": 0
}

print(f"\nProducer configuration options:")
for key, value in config_options.items():
    print(f"  {key}: {value}")

# Close producer if created
if producer:
    try:
        producer.close(timeout=0.1)
    except:
        pass
        raise

print("Example 1 completed successfully")

## Example 2: Kafka consumer setup and message consumption

In [ ]:
from kafka import KafkaConsumer
from kafka.structs import TopicPartition, OffsetAndMetadata
from kafka.consumer.subscription_state import ConsumerRebalanceListener
import json

# Example 2: Kafka consumer setup and message consumption
print("Example 2: Kafka consumer setup and message consumption")

# Create a rebalance listener for consumer group management
class CustomRebalanceListener(ConsumerRebalanceListener):
    def on_partitions_revoked(self, revoked):
        print(f"Partitions revoked: {revoked}")
    
    def on_partitions_assigned(self, assigned):
        print(f"Partitions assigned: {assigned}")

# Create rebalance listener instance
rebalance_listener = CustomRebalanceListener()
print(f"Rebalance listener created: {type(rebalance_listener).__name__}")

# Demonstrate consumer configuration (without actual connection)
consumer_config = {
    "bootstrap_servers": ['localhost:9092'],
    "group_id": 'analytics-group',
    "auto_offset_reset": 'earliest',
    "enable_auto_commit": True,
    "auto_commit_interval_ms": 5000,
    "fetch_min_bytes": 1,
    "fetch_max_wait_ms": 500
}

print(f"\nConsumer configuration:")
for key, value in consumer_config.items():
    print(f"  {key}: {value}")

# Create topic partition objects
topic_partitions = [
    TopicPartition('user-events', 0),
    TopicPartition('user-events', 1),
    TopicPartition('system-logs', 0)
]

print(f"\nTopic partitions:")
for tp in topic_partitions:
    print(f"  Topic: {tp.topic}, Partition: {tp.partition}")

# Create offset metadata for manual offset management (with leader_epoch)
offset_metadata = OffsetAndMetadata(
    offset=100,
    metadata="processed-batch-1",
    leader_epoch=5  # Required parameter for leader epoch
)
print(f"\nOffset metadata:")
print(f"  Offset: {offset_metadata.offset}")
print(f"  Metadata: {offset_metadata.metadata}")
print(f"  Leader epoch: {offset_metadata.leader_epoch}")

# Demonstrate deserializers
value_deserializer = lambda m: json.loads(m.decode('utf-8')) if m else None
key_deserializer = lambda m: m.decode('utf-8') if m else None

# Test deserializers
test_value = json.dumps({"test": "data"}).encode('utf-8')
test_key = b"test-key"
print(f"\nDeserializer test:")
print(f"  Value deserialized: {value_deserializer(test_value)}")
print(f"  Key deserialized: {key_deserializer(test_key)}")

print("Example 2 completed successfully")

## Example 3: Kafka admin client for cluster management

In [ ]:
from kafka import KafkaAdminClient
from kafka.admin import ConfigResource, ConfigResourceType, NewTopic
from kafka.structs import TopicPartition

# Example 3: Kafka admin client for cluster management
print("Example 3: Kafka admin client for cluster management")

# Demonstrate admin client configuration (without actual connection)
admin_config = {
    "bootstrap_servers": ['localhost:9092'],
    "client_id": 'admin-client-1',
    "request_timeout_ms": 30000,
    "api_version": (2, 5, 0)
}

print(f"Admin client configuration:")
for key, value in admin_config.items():
    print(f"  {key}: {value}")

# Create new topic configurations
new_topics = [
    NewTopic(
        name="user-analytics",
        num_partitions=3,
        replication_factor=1,
        topic_configs={"cleanup.policy": "delete", "retention.ms": "604800000"}  # 7 days
    ),
    NewTopic(
        name="payment-events",
        num_partitions=5,
        replication_factor=1,
        topic_configs={"cleanup.policy": "compact"}
    )
]

print(f"\nNew topics to create:")
for topic in new_topics:
    print(f"  Topic: {topic.name}")
    print(f"    Partitions: {topic.num_partitions}")
    print(f"    Replication factor: {topic.replication_factor}")
    print(f"    Configs: {topic.topic_configs}")

# Create config resources for topic configuration management
config_resources = [
    ConfigResource(ConfigResourceType.TOPIC, "user-analytics"),
    ConfigResource(ConfigResourceType.BROKER, "0")
]

print(f"\nConfig resources:")
for resource in config_resources:
    print(f"  Type: {resource.resource_type}")
    print(f"  Name: {resource.name}")

# Show admin operations that would be available with a connected client
admin_operations = [
    "create_topics", "delete_topics", "list_topics",
    "describe_configs", "alter_configs",
    "create_partitions", "describe_cluster"
]

print(f"\nAvailable admin operations (with connected client):")
for op in admin_operations:
    print(f"  ✓ {op}")

# Demonstrate ConfigResourceType enum values
print(f"\nConfig resource types:")
resource_types = ["BROKER", "TOPIC"]
for rt in resource_types:
    if hasattr(ConfigResourceType, rt):
        print(f"  {rt}: {getattr(ConfigResourceType, rt)}")

print("Example 3 completed successfully")

## Example 4: Low-level networking layer and serializer plug-in API

In [ ]:
# Example 4: Low-level networking layer + serializer plug-in API (kafka-python 3.x).
# 3.x moved the low-level client from kafka.KafkaClient / kafka.conn to kafka.net
# (KafkaNetClient / KafkaConnection / KafkaConnectionManager), and the Serializer /
# Deserializer ABCs now take (topic, headers, data).
import json

from kafka.net import KafkaNetClient, KafkaConnection, KafkaConnectionManager
from kafka.serializer import Serializer, Deserializer, JsonSerializer

print("Example 4: Low-level Kafka client and connection management")

# The low-level networking classes importable + real classes
for cls in (KafkaNetClient, KafkaConnection, KafkaConnectionManager):
    assert isinstance(cls, type)
    print(f"  {cls.__name__}: {cls.__module__}")

# Custom serializers for message encoding/decoding (3.x ABC signature)
class CustomJSONSerializer(Serializer):
    def serialize(self, topic, headers, data):
        return json.dumps(data).encode("utf-8")

class CustomJSONDeserializer(Deserializer):
    def deserialize(self, topic, headers, data):
        return json.loads(data.decode("utf-8")) if data else None

json_serializer = CustomJSONSerializer()
json_deserializer = CustomJSONDeserializer()

print(f"\nCustom serializers:")
print(f"  Serializer: {type(json_serializer).__name__}")
print(f"  Deserializer: {type(json_deserializer).__name__}")

# Demonstrate serialization round-trip
test_data = {"user_id": 123, "event": "page_view"}
serialized = json_serializer.serialize("test-topic", [], test_data)
deserialized = json_deserializer.deserialize("test-topic", [], serialized)

print(f"\nSerialization test:")
print(f"  Original: {test_data}")
print(f"  Serialized length: {len(serialized)} bytes")
print(f"  Deserialized: {deserialized}")
assert deserialized == test_data

# A stock JsonSerializer also ships in 3.x
assert issubclass(JsonSerializer, Serializer)
print(f"\nStock JsonSerializer available: {JsonSerializer.__name__}")

print("Example 4 completed successfully")

## Example 5: Kafka protocol and error handling

In [ ]:
# Example 5: Kafka protocol and error handling (kafka-python 3.x layout:
# kafka.protocol.{producer,consumer,metadata} modules + versioned ApiMessageData).
import kafka
from kafka import TopicPartition, OffsetAndMetadata
from kafka.errors import KafkaError, KafkaConnectionError, TopicAlreadyExistsError
from kafka.protocol import consumer, metadata, producer
from kafka.protocol.admin import CreateTopicsRequest, DescribeConfigsRequest

print("Example 5: Kafka protocol and error handling")
print(f"Kafka-python version: {kafka.__version__}")

# Versioned protocol request classes resolve via indexing
v3 = CreateTopicsRequest[3]
v2 = DescribeConfigsRequest[2]
print(f"\nCreateTopicsRequest: {len(CreateTopicsRequest)} versions; v3 = {v3.__name__}")
print(f"DescribeConfigsRequest v2 = {v2.__name__}")
assert v3.__name__ == "CreateTopicsRequest_v3"

print(f"\nCore protocol request classes:")
print(f"  ProduceRequest: {producer.ProduceRequest}")
print(f"  FetchRequest: {consumer.FetchRequest}")
print(f"  MetadataRequest: {metadata.MetadataRequest}")

# Error hierarchy
print(f"\nKafka error types:")
for error_type in (KafkaError, KafkaConnectionError, TopicAlreadyExistsError):
    error_doc = (error_type.__doc__ or "Kafka error").split(".")[0]
    print(f"  {error_type.__name__}: {error_doc}")
assert issubclass(TopicAlreadyExistsError, KafkaError)

# Topic partition and offset management
topic_partition = TopicPartition("metrics", 2)
offset_metadata = OffsetAndMetadata(
    offset=1500,
    metadata="batch-processed-150",
    leader_epoch=10,
)
print(f"\nTopic partition management:")
print(f"  Topic: {topic_partition.topic}")
print(f"  Partition: {topic_partition.partition}")
print(f"  Offset: {offset_metadata.offset}")
print(f"  Metadata: {offset_metadata.metadata}")
print(f"  Leader epoch: {offset_metadata.leader_epoch}")

print("Example 5 completed successfully")